# Week 2 homework: what controls a river's baseflow?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cdds-uiuc/applied-ai-earth-sciences/blob/main/week02_regression_regularization/W02_hw_camels_baseflow.ipynb)

**Due:** Monday of Week 3, 11:59 pm Central. **Points:** 20. **Estimated time:** 3–5 hours.

**Submit** the completed `.ipynb` with every cell executed, via Canvas.

Two reminders from the syllabus: the cells marked *Write this yourself* must be your own work, and the last cell asks you to declare any AI-assistant use.

The lab predicted *how much* water leaves a catchment. This homework asks a different hydrological question: *how* it leaves. The **baseflow index** (BFI) is the fraction of streamflow in the slowly varying part of the hydrograph, separated from the quick storm response by a standard digital filter (the Lyne–Hollick filter as implemented by Ladson et al., 2013). It is a proxy for flow sustained by groundwater and slow drainage, and in snowy catchments by snowmelt. A river with a high baseflow index keeps flowing through dry spells; one with a low index floods and then runs dry. Which catchment attributes control it? Climate again, or this time the soils and geology that the lab found unimportant for mean flow?

> **Data card.** *CAMELS catchment attributes (version 2.0).* Source: Addor et al. (2017); attribute files `camels_attributes_v2.0` (DOI 10.5065/D6G73C3Q), Zenodo record 15529996 (record version 1.2, which also carries the time series, DOI 10.5065/D6MW2F4D). License: CC BY 4.0; redistribution permitted with attribution. What: static attributes for 671 US catchments; features = the 27 attributes of Kratzert et al. (2019), with units as tabulated in the Week 1 lab; target this week = `baseflow_index` (dimensionless, 0–1), from `camels_hydro.txt`. Size: 0.6 MB, seven text files. Coverage: CONUS; climate and flow attributes are water-year 1990–2009 means, vegetation 2002–2014, soils and geology static. Pre-processing by instructor: none. Citation: Addor et al. (2017), *HESS* 21, 5293–5313; Kratzert et al. (2019), *HESS* 23, 5089–5110.


In [ ]:
# --- Setup: all imports, the seed, plot settings. The same block opens every notebook in this course. ---
import os
import random
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

seed = 42                      # one seed for the whole notebook; re-set it before any cell you re-run
random.seed(seed)
np.random.seed(seed)

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["font.size"] = 12
# Okabe–Ito: eight colorblind-safe colors as the default color cycle
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=["#E69F00", "#56B4E9", "#009E73", "#F0E442",
                                                     "#0072B2", "#D55E00", "#CC79A7", "#000000"])


## Provided: get and prepare the data

> **AI assistant: good use here.** These cells are given to you. Read them, run them, check the shapes.


In [ ]:
data_dir = "data"                       # created next to this notebook; ignored by git
os.makedirs(data_dir, exist_ok=True)

base_url = "https://zenodo.org/records/15529996/files/"
attribute_files = ["camels_clim", "camels_geol", "camels_hydro", "camels_name",
                   "camels_soil", "camels_topo", "camels_vege"]

for name in attribute_files:
    fname = os.path.join(data_dir, name + ".txt")
    if not os.path.exists(fname):       # download once; a hand-placed file is simply used
        print("downloading", name)
        urllib.request.urlretrieve(base_url + name + ".txt?download=1", fname)
print("all seven attribute files are in", data_dir)


In [ ]:
# The seven files share the column gauge_id. Read each one and join them side by side.
tables = []
for name in attribute_files:
    table = pd.read_csv(os.path.join(data_dir, name + ".txt"), sep=";", dtype={"gauge_id": str})
    tables.append(table.set_index("gauge_id"))
camels = pd.concat(tables, axis=1)      # one row per catchment, every attribute as a column
print(camels.shape)                     # (n_catchments, n_columns)


In [ ]:
# The 27 static attributes used by Kratzert et al. (2019). None is computed from streamflow, which matters:
# anything derived from the target (q_mean, runoff_ratio, baseflow_index, ...) would be cheating.
# (One subtlety: CAMELS' potential evaporation uses a Priestley-Taylor coefficient calibrated per catchment inside
# a rainfall-runoff model (Newman et al., 2015), so pet_mean and aridity are not purely meteorological.)
feature_names = [
    "p_mean", "pet_mean", "aridity", "p_seasonality", "frac_snow",
    "high_prec_freq", "high_prec_dur", "low_prec_freq", "low_prec_dur",
    "elev_mean", "slope_mean", "area_gages2",
    "frac_forest", "lai_max", "lai_diff", "gvf_max", "gvf_diff",
    "soil_depth_pelletier", "soil_depth_statsgo", "soil_porosity", "soil_conductivity",
    "max_water_content", "sand_frac", "silt_frac", "clay_frac",
    "carbonate_rocks_frac", "geol_permeability",
]
target_name = "baseflow_index"          # fraction of flow that is baseflow (dimensionless, 0-1)

catchments = camels.dropna(subset=feature_names + [target_name])     # baseflow_index has no gaps: all 671 catchments, one more than the lab

def rmse(y: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y - y_pred) ** 2)))

X = catchments[feature_names].values
y = catchments[target_name].values
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=seed)
print(len(catchments), "catchments;", X_train.shape, X_val.shape, X_test.shape)
print(f"baseflow index ranges from {y.min():.2f} to {y.max():.2f} (median {np.median(y):.2f})")


In [ ]:
fig, ax = plt.subplots()
ax.hist(y, bins=40)
ax.set_xlabel("baseflow index [-]")
ax.set_ylabel("number of catchments")
ax.set_title("The target: baseflow index of the 671 catchments")
plt.show()


A roughly symmetric distribution centered near 0.5, from almost 0 (flashy, storm-fed rivers) to almost 1 (rivers fed almost entirely by slow drainage); half the catchments lie between 0.40 and 0.60.


## Question 1 (5 points): baseline and plain linear regression

Standardize the features with the training set's statistics (`StandardScaler` fitted on `X_train` only, applied to all three sets); call the results `X_train_std`, `X_val_std` and `X_test_std`, which the self-check and the later questions use. Report the test RMSE of the mean-of-training-set baseline (`rmse_baseline`) and of a 27-feature `LinearRegression` (`rmse_ols`), both in units of the baseflow index (dimensionless).

Deliverable: two printed numbers, labeled (the print lines are provided).

> **Write this yourself.**

### Answer


In [ ]:
# YOUR CODE HERE
print(f"baseline test RMSE: {rmse_baseline:.4f}   (units of the baseflow index)")
print(f"OLS test RMSE:      {rmse_ols:.4f}")


In [ ]:
# Self-check: run this cell. It prints only the last line if all is well.
assert X_train_std.shape[1] == 27, "standardized features should still have 27 columns"
assert abs(X_train_std.mean()) < 1e-6, "standardized training features should have mean zero"
assert 0 < rmse_ols < rmse_baseline, "the linear model should beat the baseline"
print("All checks passed.")


## Question 2 (5 points): Ridge, with alpha chosen on the validation set

Sweep `alpha` over `np.logspace(-2, 3, 30)`, fitting `Ridge` on the standardized training set and scoring on the validation set. Plot validation RMSE against alpha (log x-axis, labeled axes), mark the best alpha, refit at that alpha, and report its test RMSE. Call the chosen alpha `best_alpha_ridge` and the test RMSE `rmse_ridge` (the print line uses them).

Deliverable: the plot, the chosen alpha, and the test RMSE.

> **Write this yourself.**

### Answer


In [ ]:
alphas = np.logspace(-2, 3, 30)
# YOUR CODE HERE
print(f"best alpha = {best_alpha_ridge:.3g};  Ridge test RMSE: {rmse_ridge:.4f}")


In [ ]:
# Self-check
assert best_alpha_ridge in alphas, "best_alpha_ridge should be one of the swept values"
assert rmse_ridge > 0
print("All checks passed.")


## Question 3 (5 points): Lasso selects the attributes

Sweep `alpha` over `np.logspace(-4, -1, 30)` for `Lasso(max_iter=50000)` on the standardized features and record the validation RMSE at each alpha. As in the lab, the validation curve will be flat, so apply the parsimony rule: choose the **largest alpha whose validation RMSE is within 5 % of the minimum**. Refit at that alpha and report the test RMSE and the list of attributes with non-zero coefficients. Make a horizontal bar chart of the kept coefficients, sorted, with labeled axes. Names the print lines and self-check use: store the kept coefficients in a pandas Series `kept`, indexed by attribute name and sorted by value as in the lab; the parsimonious alpha in `alpha_parsimonious`; the test RMSE in `rmse_lasso`; and, for comparison, the number of attributes the validation-minimum alpha keeps in `n_kept_at_minimum`.

Deliverable: the bar chart, the number of attributes kept at the two alphas, and the names of the kept attributes.

> **Write this yourself.**

### Answer


In [ ]:
alphas_lasso = np.logspace(-4, -1, 30)
# YOUR CODE HERE
print(f"validation-minimum alpha keeps {n_kept_at_minimum} of 27 attributes")
print(f"parsimonious alpha = {alpha_parsimonious:.3g};  Lasso test RMSE: {rmse_lasso:.4f};  {len(kept)} of 27 attributes kept:")
print(kept.round(4).to_string())


In [ ]:
# Self-check
assert isinstance(kept, pd.Series), "kept should be a pandas Series indexed by attribute name"
assert 1 <= len(kept) <= 20, "the parsimony rule should leave a short list (1-20 attributes)"
assert set(kept.index) <= set(feature_names)
assert n_kept_at_minimum >= len(kept), "the validation-minimum alpha should keep at least as many attributes as the parsimonious one"
print("All checks passed.")


## Question 4 (5 points): interpret

One or two sentences per part, labeled (a), (b) and (c).

(a) Did regularization lower the test RMSE compared with plain linear regression, and did Ridge or Lasso do better? If the answer is "neither helped", say so and say what the numbers show. (2 points)

(b) Compare the attributes Lasso kept for the baseflow index with those the lab's Lasso kept for mean flow, comparing coefficient sizes relative to the largest coefficient rather than just counting. Does the baseflow index depend more on soils and geology, as the physical picture (slow drainage through the subsurface) would suggest? (2 points)

(c) Name one thing a linear model with these 27 attributes cannot capture about baseflow. (1 point)

> **Write this yourself.**

### Answer

*Your interpretation here.*


## AI-use declaration

The syllabus requires this declaration on every homework. Fill in one row per tool you used (add rows as needed); write "none" in the first row if you used no assistant.

| Tool | Cells (question numbers) | What you asked it for | What you checked |
|---|---|---|---|
| | | | |
